In [52]:
import torch
import pandas as pd

## Dataset Loading

In [53]:
train_df=pd.read_csv('../data/train.csv')
test_df=pd.read_csv('../data/test.csv')

## Data Cleaning and Pre-Processing

In [54]:
train_df["HasCabin"]=~train_df["Cabin"].apply(lambda x: pd.isnull(x))
test_df["HasCabin"]=~test_df["Cabin"].apply(lambda x: pd.isnull(x))

test_df["Fare"]=test_df["Fare"].fillna(test_df["Fare"].mean())

train_df['Age']=train_df.groupby(['Pclass','Sex'])['Age'].transform(lambda x:x.fillna(x.median()))
test_df['Age']=test_df.groupby(['Pclass','Sex'])['Age'].transform(lambda x:x.fillna(x.median()))

train_df['FamilySize']=train_df['SibSp']+train_df['Parch']+1
test_df['FamilySize']=test_df['SibSp']+test_df['Parch']+1

train_df['IsAlone']=(train_df['FamilySize']==1).astype(int)
test_df['IsAlone']=(test_df['FamilySize']==1).astype(int)

train_df['Embarked']=train_df['Embarked'].fillna(train_df['Embarked'].mode()[0])

train_df['Title']=train_df['Name'].str.extract(r',\s*([^\.]+)\.')
test_df['Title']=test_df['Name'].str.extract(r',\s*([^\.]+)\.')

train_df['Title']=train_df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})
test_df['Title']=test_df['Title'].replace({'Mlle':'Miss','Ms':'Miss','Mme':'Mrs'})

title_counts=train_df['Title'].value_counts()
rare_titles=title_counts[title_counts<10].index

train_df['Title']=train_df['Title'].apply(lambda x:'Rare' if x in rare_titles else x)
test_df['Title']=test_df['Title'].apply(lambda x:'Rare' if x in rare_titles else x)

train_df["Sex"]=train_df["Sex"].map({"male":1,"female":0})
test_df["Sex"]=test_df["Sex"].map({"male":1,"female":0})

combined = pd.concat([train_df, test_df], sort=False)
combined = pd.get_dummies(combined, columns=['Embarked', 'Title'], drop_first=True)

train_df = combined.iloc[:len(train_df)].copy()
test_df = combined.iloc[len(train_df):].drop(columns=['Survived']).copy()

train_df.drop(["Cabin","Name","Ticket","PassengerId"],inplace=True,axis=1)
test_df.drop(["Cabin","Name","Ticket"],inplace=True,axis=1)


In [74]:
print(train_df.info())
print(set(train_df.columns)-set(test_df.columns),set(test_df.columns)-set(train_df.columns))

<class 'pandas.DataFrame'>
Index: 891 entries, 0 to 890
Data columns (total 17 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Survived      891 non-null    float64
 1   Pclass        891 non-null    int64  
 2   Sex           891 non-null    int64  
 3   Age           891 non-null    float64
 4   SibSp         891 non-null    int64  
 5   Parch         891 non-null    int64  
 6   Fare          891 non-null    float64
 7   HasCabin      891 non-null    bool   
 8   FamilySize    891 non-null    int64  
 9   IsAlone       891 non-null    int64  
 10  Embarked_Q    891 non-null    bool   
 11  Embarked_S    891 non-null    bool   
 12  Title_Master  891 non-null    bool   
 13  Title_Miss    891 non-null    bool   
 14  Title_Mr      891 non-null    bool   
 15  Title_Mrs     891 non-null    bool   
 16  Title_Rare    891 non-null    bool   
dtypes: bool(8), float64(3), int64(6)
memory usage: 76.6 KB
None
{'Survived'} {'PassengerId'}


## DataFrame to DataLoader

In [56]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from torch import nn
from torch.utils.data import TensorDataset,DataLoader

device=torch.device("cuda" if torch.cuda.is_available else "cpu")
x=train_df.drop(["Survived"],axis=1).values
y=train_df["Survived"].values
x=x.astype(float)
y=y.astype(float)

x=torch.tensor(x,dtype=torch.float32)
y=torch.tensor(y,dtype=torch.long)

train_x, test_x, train_y, test_y = train_test_split(x, y, test_size=0.1, random_state=50, stratify=y)
train_dataset=TensorDataset(train_x,train_y)
test_dataset=TensorDataset(test_x,test_y)
train_loader=DataLoader(train_dataset,shuffle=True,batch_size=32)
test_loader=DataLoader(test_dataset,batch_size=32)

## Custom Logistic Regression Model

In [57]:
class Model(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1=nn.Linear(16,64)
        self.layer2=nn.Linear(64,64)
        self.layer3=nn.Linear(64,2)
        self.sigmoid=nn.SiLU()
    def forward(self,x):
        x=self.layer1(x)
        x=self.sigmoid(x)
        x=self.layer2(x)
        x=self.sigmoid(x)
        x=self.layer3(x)
        return x

## Training Custom Model

In [ ]:
epoch=100
model=Model()
model=model.to(device)
optimizer=torch.optim.SGD(model.parameters(),lr=0.0005,weight_decay=0.0001)
loss_fn=nn.CrossEntropyLoss()
best=0
for i in range(epoch):
    model.train()
    batch_loss=0
    for data,result in train_loader:
        data=data.to(device)
        result=result.to(device)
        predicted_result=model(data)
        loss=loss_fn(predicted_result,result)
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        batch_loss+=loss.item()
    model.eval()
    correct=0
    total=0
    for data,result in test_loader:
        data=data.to(device)
        result=result.to(device)
        predicted_result=model(data)
        correct+=(predicted_result.argmax(1)==result).type(torch.float).sum().item()
        total+=result.size(0)
    
    if i%20==0:
        print(f"epoch {i} loss {batch_loss/len(train_loader)} validation accuracy {correct/total*100}%")
        best=max(best,correct/total*100)
print(best)


## Logistic Regression

In [58]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score

lr=LogisticRegression(max_iter=1000)
lr_scores=cross_val_score(lr,x,y,cv=5,scoring='accuracy')
print("Logistic Regression CV accuracy:",lr_scores.mean())

Logistic Regression CV accuracy: 0.824913690289373


## Random Forest Classifier

In [59]:
from sklearn.ensemble import RandomForestClassifier
rf=RandomForestClassifier(n_estimators=200,random_state=50)
rf_scores=cross_val_score(rf,x,y,cv=5,scoring='accuracy')
print("Random Forest CV accuracy:",rf_scores.mean())

Random Forest CV accuracy: 0.809189630280585


## Gradient Boosting Classifier

In [60]:
from sklearn.ensemble import GradientBoostingClassifier
gb=GradientBoostingClassifier(random_state=50)
gb_scores=cross_val_score(gb,x,y,cv=5,scoring='accuracy')
print("Gradient Boosting CV accuracy:",gb_scores.mean())

Gradient Boosting CV accuracy: 0.827154604230745


## Final Model of Titanic prediction

In [ ]:
X_df = train_df.drop(["Survived"], axis=1).astype(float)
y_df = train_df["Survived"].astype(float)

predict_test_df = test_df.copy()
predict_test_df.drop(["PassengerId"], inplace=True, axis=1)
predict_test_df = predict_test_df.astype(float)

model = LogisticRegression(max_iter=1000)
model.fit(X_df, y_df)
test_predictions = model.predict(predict_test_df)

submission = pd.DataFrame({
    "PassengerId": test_df["PassengerId"],
    "Survived": test_predictions.astype(int)
})
submission.to_csv("submission.csv", index=False)